In [1]:
import reframed
from pathlib import Path
import pandas as pd
import json
import numpy as np
from reframed.solvers.solution import Status

import sys
sys.path.append('../../code/7_GEM_reconstruction')

from ng_utils import *
from ng_tests import *
from model_qc_and_polish import curate_model, remove_duplicated_metabolites, remove_duplicated_reactions, test_balance

# Evaluate model performance

In [111]:
repo_folder = Path("../..")
data_folder = repo_folder / "data" / "7_GEM_reconstruction"
carveme_draft_folder = data_folder / 'models/carveme'
polished_folder = data_folder / 'models/polished'
growth_data_folder = repo_folder / 'data' / '1_growth_phenotyping'
binary_growth_data_path = growth_data_folder / 'growth_no_growth.csv'
carbon_source_id_path = growth_data_folder / 'selected_carbon_sources.csv'#'carbon_source_ids_curated.csv'

In [112]:
auxotrophy_dict = {
'Ml': {'amino acids': ['cys__L'], 'vitamins':['thm', 'btn']},#'pro__L',
'Oa': {'vitamins': ['thm']}
}
auxotrophy_constraints = set_auxotrophy_dict(auxotrophy_dict)

In [113]:
model_abbrvs = ['At', 'Ct', 'Ml', 'Oa']
model_dict = {}
for abbrv in model_abbrvs:
    model_path = polished_folder / f'{abbrv}.xml'
    model = reframed.load_cbmodel(str(model_path))
    model_dict[abbrv] = model

## At

In [115]:
at = model_dict['At']
sol = reframed.FBA(at, constraints = {'R_EX_glc__D_e':0})#'R_MECDPDH4E':0 'R_MECDPDH4E':0
print(sol.fobj)
sol.show_values()

None


# Ct

In [116]:
ct = model_dict['Ct']
sol = reframed.FBA(ct, constraints = {'R_EX_ac_e':0})#'R_MECDPDH4E':0 'R_MECDPDH4E':0
print(sol.fobj)
sol.show_values()

None


# Ml

In [117]:
ml = model_dict['Ml']
sol = reframed.FBA(ml, constraints = {'R_EX_glc__D_e':0})#'R_MECDPDH4E':0 'R_MECDPDH4E':0
print(sol.fobj)
sol.show_values()

None


# Oa

In [118]:
oa = model_dict['Oa']
sol = reframed.FBA(oa, constraints = {'R_EX_glc__D_e':0})#'R_MECDPDH4E':0 'R_MECDPDH4E':0
print(sol.fobj)
sol.show_values()

None


# Test growth on carbon sources

In [119]:
growth_data = pd.read_csv(binary_growth_data_path, index_col=0)
cs_name_to_id = pd.read_csv(carbon_source_id_path, index_col=0).to_dict()[f'BiGG ID']

In [120]:
test_dfs = {}
pred_matrices = {}
for abbrv, model in model_dict.items():
    test_df, pred_matrix = test_model(model, growth_data[abbrv], cs_name_to_id, 
            additional_constraints=auxotrophy_constraints.get(abbrv, {}))
    test_dfs[abbrv] = test_df
    pred_matrices[abbrv] = pred_matrix


Change lower bound of R_EX_glc__D_e from -10.0 to 0
{'R_EX_ac_e': -10, 'R_EX_glc__D_e': 0}
{'R_EX_adn_e': -10, 'R_EX_glc__D_e': 0}
{'R_EX_ala__L_e': -10, 'R_EX_glc__D_e': 0}
{'R_EX_arab__D_e': -10, 'R_EX_glc__D_e': 0}
{'R_EX_arg__L_e': -10, 'R_EX_glc__D_e': 0}
{'R_EX_asp__L_e': -10, 'R_EX_glc__D_e': 0}
{'R_EX_cit_e': -10, 'R_EX_glc__D_e': 0}
{'R_EX_cys__L_e': -10, 'R_EX_glc__D_e': 0}
{'R_EX_for_e': -10, 'R_EX_glc__D_e': 0}
{'R_EX_fum_e': -10, 'R_EX_glc__D_e': 0}
{'R_EX_glc__D_e': -10}
{'R_EX_glu__L_e': -10, 'R_EX_glc__D_e': 0}
{'R_EX_gln__L_e': -10, 'R_EX_glc__D_e': 0}
{'R_EX_glutar_e': -10, 'R_EX_glc__D_e': 0}
{'R_EX_glyc_e': -10, 'R_EX_glc__D_e': 0}
{'R_EX_gly_e': -10, 'R_EX_glc__D_e': 0}
{'R_EX_his__L_e': -10, 'R_EX_glc__D_e': 0}
{'R_EX_ins_e': -10, 'R_EX_glc__D_e': 0}
{'R_EX_ile__L_e': -10, 'R_EX_glc__D_e': 0}
{'R_EX_lac__L_e': -10, 'R_EX_glc__D_e': 0}
{'R_EX_leu__L_e': -10, 'R_EX_glc__D_e': 0}
{'R_EX_mal__L_e': -10, 'R_EX_glc__D_e': 0}
{'R_EX_mnl_e': -10, 'R_EX_glc__D_e': 0}
{'R_E

In [121]:
df = pd.concat(test_dfs).reset_index(drop=True)

# Test auxotrophies

## Ml

In [127]:
print(reframed.FBA(ml).fobj)
for r_id in ['R_EX_glc__D_e', 'R_EX_cys__L_e', 'R_EX_thm_e', 'R_EX_btn_e']:
    sol = reframed.FBA(ml, constraints = {r_id:0})#'R_MECDPDH4E':0 'R_MECDPDH4E':0
    print(r_id, sol.fobj)

0.8393838922231084
R_EX_glc__D_e None
R_EX_cys__L_e -0.0
R_EX_thm_e -0.0
R_EX_btn_e -0.0


## Oa

In [128]:
print(reframed.FBA(oa).fobj)
for r_id in ['R_EX_glc__D_e', 'R_EX_thm_e']:
    sol = reframed.FBA(oa, constraints = {r_id:0})#'R_MECDPDH4E':0 'R_MECDPDH4E':0
    print(r_id, sol.fobj)

0.9790075808679629
R_EX_glc__D_e None
R_EX_thm_e -0.0


In [123]:
df.to_csv(data_folder / 'model_performance_summary.csv', index=False)

In [122]:
pred_matrices

{'At': defaultdict(int, {'TP': 29, 'TN': 4}),
 'Ct': defaultdict(int, {'TP': 19, 'TN': 14}),
 'Ml': defaultdict(int, {'TP': 32, 'TN': 1}),
 'Oa': defaultdict(int, {'TP': 27, 'TN': 6})}

# Testing below

In [84]:
model = at

In [102]:
sol = reframed.FBA(model, constraints = {'R_EX_for_e':0, 'R_EX_glc__D_e':0, 'R_EX_ac_e':0, 'R_EX_cys__L_e':0, 'R_BTS2':0})#'R_MECDPDH4E':0 'R_MECDPDH4E':0
print(sol.fobj)
sol.show_values()

None


In [86]:
sol.show_metabolite_balance('M_h_c', model)

[ --> o ] R_ATPS4rpp    47.25    
[ --> o ] R_CYSS        6.3      
[ --> o ] R_PGCD        6.3      
[ --> o ] R_PPDK        6.3      
[ --> o ] R_ATPM        3.15     
[ o --> ] R_CYOO2pp    -25.2     
[ o --> ] R_CYTMQOR3pp -6.3      
[ o --> ] R_NADH17pp   -25.2     
[ <-- o ] R_BTS2       -12.6     


In [91]:
sol.show_metabolite_balance('M_focytC_c', model)

[ --> o ] R_CYTMQOR3pp  12.6     
[ o --> ] R_CYOO2pp    -12.6     


In [101]:
model.reactions['R_BTS2'],  sol.values['R_BTS2']

(R_BTS2: M_cys__L_c + M_dtbt_c <-> M_ala__L_c + M_btn_c + 2.0 M_h_c,
 -6.300000000000001)

In [96]:
model.reactions['R_GGGABADxr'], sol.values['R_GGGABADxr']

(R_GGGABADxr: M_ggbutal_c + M_h2o_c + M_nad_c <-> M_gg4abut_c + 2.0 M_h_c + M_nadh_c,
 0.0)

In [ ]:
print(model.reactions['R_NADH18pp'])
print(sol.values['R_NADH18pp'])

R_NADH18pp: M_2dmmq8_c + 4.0 M_h_c + M_nadh_c --> M_2dmmql8_c + 3.0 M_h_p + M_nad_c
1.8


In [ ]:
sol.show_metabolite_turnover(model, 'M_atp_c')

M_atp_c       3.15


In [ ]:
# r_metabolites = r.get_products() + r.get_substrates()

In [ ]:
sol = reframed.FBA(model, shadow_prices=True, reduced_costs=True)

In [ ]:
balance_df = test_balance(model)

In [ ]:
idx = (~balance_df.mass_balance)|(~balance_df.charge_balance)|(~balance_df.element_balance)
balance_df[idx]

,r_id,mass_balance,mass,element_balance,element_dict,charge_balance,charge
16,R_23CTI1,True,4.033232e-17,True,"{'C': 0.0, 'H': 0.0, 'N': 0.0, 'O': 0.0, 'P': ...",False,1.0
165,R_ACOAD10f,True,2.220446e-16,True,"{'C': 0.0, 'H': 0.0, 'N': 0.0, 'O': 0.0, 'P': ...",False,-2.0
167,R_ACOAD12f,True,0.000000e+00,True,"{'C': 0.0, 'H': 0.0, 'N': 0.0, 'O': 0.0, 'P': ...",False,-2.0
169,R_ACOAD14f,True,1.110223e-16,True,"{'C': 0.0, 'H': 0.0, 'N': 0.0, 'O': 0.0, 'P': ...",False,-2.0
170,R_ACOAD15f,True,1.110223e-16,True,"{'C': 0.0, 'H': 0.0, 'N': 0.0, 'O': 0.0, 'P': ...",False,-2.0
171,R_ACOAD16f,True,2.220446e-16,True,"{'C': 0.0, 'H': 0.0, 'N': 0.0, 'O': 0.0, 'P': ...",False,-2.0
172,R_ACOAD17f,True,2.220446e-16,True,"{'C': 0.0, 'H': 0.0, 'N': 0.0, 'O': 0.0, 'P': ...",False,-4.0
173,R_ACOAD18f,True,2.220446e-16,True,"{'C': 0.0, 'H': 0.0, 'N': 0.0, 'O': 0.0, 'P': ...",False,-2.0
183,R_ACOAD24f,True,0.000000e+00,True,"{'C': 0.0, 'H': 0.0, 'N': 0.0, 'O': 0.0, 'P': ...",False,-1.0
191,R_ACOAD30f,True,1.110223e-16,True,"{'C': 0.0, 'H': 0.0, 'N': 0.0, 'O': 0.0, 'P': ...",False,-1.0


In [ ]:
sol.show_values('R_EX_*')

R_EX_5drib_e  4.91844e-06
R_EX_ca2_e   -0.00619723
R_EX_cl_e    -0.00619723
R_EX_co2_e    9.20029
R_EX_cobalt2_e -0.000119272
R_EX_cu2_e   -0.000844742
R_EX_fe2_e   -0.00799492
R_EX_fe3_e   -0.00929708
R_EX_glc__D_e -10
R_EX_h2o_e    73.5716
R_EX_h_e     -50
R_EX_k_e     -0.232407
R_EX_mg2_e   -0.0103287
R_EX_mn2_e   -0.000822609
R_EX_nh4_e   -13.3321
R_EX_o2_e    -22.3515
R_EX_pi_e    -1.1741
R_EX_so4_e   -0.298415
R_EX_zn2_e   -0.000405771


In [ ]:
sol.values['R_EX_glc__D_e']

-10.0

In [ ]:
df = pd.Series(sol.reduced_costs)

In [ ]:
data = []
for r in model.reactions.values():
    if np.abs(sol.values[r.id]) > 1:
        tempsol = reframed.FBA(model, constraints={r.id: (0, 0)})
        if tempsol.status != Status.OPTIMAL:
            print(f'Reaction {r.id} is essential for growth.')
            data.append([r.id, 0,0,0])
        else:
            data.append([r.id, tempsol.values['R_EX_h_e'], tempsol.values['R_EX_co2_e'], tempsol.fobj])

In [ ]:
df = pd.DataFrame(data, columns=['reaction_id', 'h_excretion', 'co2_excretion', 'biomass_flux'])

In [ ]:
df.to_csv(data_folder / f'{model_abbrv}_reaction_essentiality_analysis.csv', index=False)

In [ ]:
m.id

NameError: name 'm' is not defined

In [ ]:
sol.show_metabolite_balance('M_nadh_c', model, sort=True)

[ --> o ] R_MDH         9.78887  
[ --> o ] R_AKGDH       2.03369  
[ --> o ] R_ACOAD7      0.564052 
[ --> o ] R_PGCD        0.415403 
[ --> o ] R_IPMD        0.12647  
[ --> o ] R_IMPD        0.0682776
[ --> o ] R_HDH         0.0531887
[ --> o ] R_PPND        0.0387721
[ --> o ] R_GLYCL       0.015273 
[ --> o ] R_GCALDD      0.000250472
[ --> o ] R_E4PD        6.26179e-05
[ --> o ] R_PDX5PS      6.26179e-05
[ --> o ] R_PERD        6.26179e-05
[ o --> ] R_NADH6      -9.28943  
[ <-- o ] R_GDH1       -2.52055  
[ <-- o ] R_GAPD       -0.985438 
[ o --> ] R_IPDPS      -0.30901  


In [ ]:
model.reactions.R_ACOAD7

R_ACOAD7: M_nad_c + M_pmtcoa_c <-> M_h_c + M_hdd2coa_c + M_nadh_c

In [ ]:
ecoli_fn = repo_folder /  "data" / "7_GEM_reconstruction" / 'iML1515.xml'

In [ ]:
ecoli = reframed.load_cbmodel(str(ecoli_fn))

In [ ]:
sole = reframed.FBA(ecoli)
sole.show_values('R_EX_*')

R_EX_pi_e    -0.845957
R_EX_co2_e    24.0033
R_EX_h_e      8.0582
R_EX_mn2_e   -0.000606005
R_EX_fe2_e   -0.0140855
R_EX_glc__D_e -10
R_EX_zn2_e   -0.000299056
R_EX_mg2_e   -0.00760795
R_EX_ca2_e   -0.00456477
R_EX_ni2_e   -0.00028327
R_EX_meoh_e   1.75399e-06
R_EX_cu2_e   -0.000621791
R_EX_cobalt2_e -2.19249e-05
R_EX_h2o_e    47.1624
R_EX_mobd_e  -6.13898e-06
R_EX_so4_e   -0.220845
R_EX_nh4_e   -9.4715
R_EX_k_e     -0.171184
R_EX_cl_e    -0.00456477
R_EX_o2_e    -22.1318


In [ ]:
sole.show_metabolite_balance('M_nadh_c', ecoli, sort=True)

[ --> o ] R_GAPD        17.1054  
[ --> o ] R_PDH         9.68969  
[ --> o ] R_MDH         6.88459  
[ --> o ] R_AKGDH       5.97006  
[ --> o ] R_PGCD        1.50654  
[ --> o ] R_IPMD        0.395114 
[ --> o ] R_IMPD        0.21331  
[ --> o ] R_HISTD       0.16617  
[ --> o ] R_PPND        0.12113  
[ --> o ] R_GLYCL       0.0475407
[ --> o ] R_SHCHD2      0.00019557
[ --> o ] R_PDX5PS      0.00019557
[ --> o ] R_PDX5PO2     0.00019557
[ --> o ] R_AMPMS2      0.00019557
[ --> o ] R_OPHHX3      0.00019557
[ o --> ] R_NADH16pp   -37.997   
[ o --> ] R_FADRx      -1.91702  
[ o --> ] R_HACD4      -0.312106 
[ o --> ] R_HACD5      -0.312106 
[ o --> ] R_HACD1      -0.312106 
[ o --> ] R_HACD2      -0.312106 
[ o --> ] R_HACD3      -0.312106 
[ o --> ] R_HACD6      -0.243854 
[ o --> ] R_HACD7      -0.243854 
[ o --> ] R_MTHFR2     -0.135372 
[ o --> ] R_IPDPS      -0.00209515
[ <-- o ] R_E4PD       -0.000391141
[ <-- o ] R_PERD       -0.000391141


In [ ]:
import cobra

In [ ]:
polished_path = data_folder / 'models/polished' / f'{model_abbrv}.xml'
modelc = cobra.io.read_sbml_model(str(polished_path))

Adding exchange reaction EX_12ppd__R_e with default bounds for boundary metabolite: 12ppd__R_e.
Adding exchange reaction EX_15dap_e with default bounds for boundary metabolite: 15dap_e.
Adding exchange reaction EX_23camp_e with default bounds for boundary metabolite: 23camp_e.
Adding exchange reaction EX_23ccmp_e with default bounds for boundary metabolite: 23ccmp_e.
Adding exchange reaction EX_23cgmp_e with default bounds for boundary metabolite: 23cgmp_e.
Adding exchange reaction EX_23cump_e with default bounds for boundary metabolite: 23cump_e.
Adding exchange reaction EX_25dkglcn_e with default bounds for boundary metabolite: 25dkglcn_e.
Adding exchange reaction EX_26dap__M_e with default bounds for boundary metabolite: 26dap__M_e.
Adding exchange reaction EX_2ameph_e with default bounds for boundary metabolite: 2ameph_e.
Adding exchange reaction EX_2m35mdntha_e with default bounds for boundary metabolite: 2m35mdntha_e.
Adding exchange reaction EX_2pglyc_e with default bounds for b

In [ ]:
modelc.reactions.KAT18.check_mass_balance()

{}

In [ ]:
for m_id in r_metabolites:
    met = modelc.metabolites.get_by_id(m_id[2:])
    print(met.id, met.charge, r.stoichiometry[m_id])

KeyError: 'M_actp_c'

In [ ]:
for m_id in r_metabolites:
    met = model.metabolites[m_id]
    print(met.id, met.metadata['CHARGE'], r.stoichiometry[m_id])
    charge = int(float(met.metadata['CHARGE']))
    met.metadata['CHARGE'] = charge


M_actp_c -2 1.0
M_g3p_c -2.0 1.0
M_h2o_c 0 1.0
M_pi_c -2 -1.0
M_xu5p__D_c -2 -1.0


In [ ]:
for 

-2

In [ ]:
for met in r.metabolites:
    print(met.id, met.charge * r.stoichiometry[met])

D-Xylulose 5-phosphate

In [ ]:
def set_auxotrophy_dict(auxotrophy_dict):
    constraints_dict = {}
    for species_abbr, species_dict in auxotrophy_dict.items():
        species_constraint_dict = {}
        if species_dict.get('vitamins'):
            for met_id in species_dict['vitamins']:
                r_id = f'R_EX_{met_id}_e'
                species_constraint_dict[r_id] = (-0.001, 0)
        if species_dict.get('amino acids'):
            for met_id in species_dict['amino acids']:
                r_id = f'R_EX_{met_id}_e'
                species_constraint_dict[r_id] = (-0.2, 0)
        constraints_dict[species_abbr] = species_constraint_dict
    return constraints_dict

In [ ]:
repo_path = Path('/Users/ssulheim/git/mwf_gems')
carveme_draft_folder = repo_path / 'models/carveme'
binary_growth_data_path = repo_path / 'data'/'growth_no_growth.csv'
carbon_source_ids_path = repo_path / 'data'/'carbon_source_ids_curated.csv'

gapfilling_data_folder = repo_path / 'gapfilling_data'
M9_minimal_media_file = gapfilling_data_folder / 'M9_minimal_media_bigg.csv'
vitamins_file = gapfilling_data_folder / 'vitamins_bigg.csv'
bigg_universe_fn = gapfilling_data_folder / 'universe_bacteria.xml'#'bigg_universe.xml'
TMP_FOLDER = repo_path  / 'tmp'

In [ ]:
binary_growth_data = pd.read_csv(binary_growth_data_path, index_col=0)
cs_name_to_id = pd.read_csv(carbon_source_ids_path, index_col=1).to_dict()['BiGG ID']
universe = reframed.load_cbmodel(bigg_universe_fn)

# Growth phenotypes

In [ ]:
auxotrophy_dict = {
    'Ml': {'amino acids': ['pro__L', 'cys__L'], 'vitamins':['thm', 'btn']},
    'Oa': {'vitamins': ['thm']}
    }
constraints_dict = set_auxotrophy_dict(auxotrophy_dict)

test_dfs = {}
pred_matrices = {}
models = {}
model_folder = repo_path / 'models' / 'gf_models' / 'carveme_FBA'
for species_abbr in ['At', 'Ct', 'Oa', 'Ml']:
    print(species_abbr)
    model_fn =  model_folder / f'gapfilled_{species_abbr}.xml'
    # model_fn = carveme_draft_folder / f'{species_abbr}.xml'
    model = reframed.load_cbmodel(model_fn)
    models[species_abbr] = model
    sol = reframed.FBA(model)
    # sol.show_values('R_EX_*')
    # model.reactions['R_EX_cit_e'].lb = 0
    # model.reactions['R_EX_glc__D_e'].lb = 0
    # Prep model
    # curate_model(model)
    # msg = remove_duplicated_metabolites(model)
    species_growth_data = binary_growth_data[species_abbr]
    if constraints_dict.get(species_abbr):
        auxotrophy_constraints = constraints_dict[species_abbr]
    else:
        auxotrophy_constraints = {}
    test_df, pred_matrix = test_model(model, species_growth_data, cs_name_to_id, min_growth=0.1, additional_constraints=auxotrophy_constraints)
    test_df['Species'] = species_abbr
    # print(pred_matrix)
    # print(MCC(pred_matrix))
    # MCC_dict[species_abbr] = MCC(pred_matrix)
    test_dfs[species_abbr] = test_df
    pred_matrices[species_abbr] = pred_matrix
    

At
Set parameter Username
Academic license - for non-commercial use only - expires 2024-02-26
Change lower bound of R_EX_glc__D_e from -10.0 to 0
Ct
Change lower bound of R_EX_cit_e from -10.0 to 0
Oa
Change lower bound of R_EX_glc__D_e from -10.0 to 0
Ml
Change lower bound of R_EX_glc__D_e from -10.0 to 0


## Make report

In [ ]:
df = pd.concat(test_dfs).reset_index(drop=True)

In [ ]:
df.to_csv('growth_prediction_table.csv')


In [ ]:
data = []
for key, pm in pred_matrices.items():
    pmd = defaultdict(int)
    pmd.update(pm)
    data.append([key, precision(pmd), accuracy(pmd), recall(pmd), f1_score(pmd), MCC(pmd)])

In [ ]:
df2 = pd.DataFrame(data, columns = ['Species abbrv.', 'Precision', 'Accuracy', 'Recall', 'F1 score', 'MCC'])
df2.to_csv('growth_prediction_scores.csv')

In [ ]:
df2

,Species abbrv.,Precision,Accuracy,Recall,F1 score,MCC
0,At,1.0,1.0,1.0,1.0,1.0
1,Ct,1.0,1.0,1.0,1.0,1.0
2,Oa,1.0,1.0,1.0,1.0,1.0
3,Ml,1.0,1.0,1.0,1.0,1.0


In [ ]:
def precision(pmd):
    return pmd['TP'] / (pmd['FP'] + pmd['TP'])
def accuracy(pmd):
    return (pmd['TP'] + pmd['TN'])/ (pmd['TP'] + pmd['FN'] + pmd['TN'] + pmd['FP'])
def recall(pmd):
    return pmd['TP'] / (pmd['FN'] + pmd['TP'])
def f1_score(pmd):
    p = precision(pmd)
    r = recall(pmd)
    return 2*p*r/(p+r)


# Test auxotrophies

In [ ]:
aux_data = []
cs_id_to_name = {value: key for key, value in cs_name_to_id.items()}
cs_id_to_name['thm'] = 'Thiamin'
cs_id_to_name['btn'] = 'Biotin'

for species_abbr, auxotrophy_constraints in constraints_dict.items():
    print(species_abbr, auxotrophy_constraints)
    model = models[species_abbr]
    for r_id, _ in auxotrophy_constraints.items():
        sol = reframed.FBA(model, constraints={r_id:0})
        m_id = r_id.lstrip('R_EX_').rstrip('_e')
        m_name = cs_id_to_name[m_id]
        aux_data.append([species_abbr, m_name, sol.fobj > 0.1])
aux_df = pd.DataFrame(aux_data, columns = ['Species abbrv.', 'Metabolite', 'Growth'])

Ml {'R_EX_thm_e': (-0.001, 0), 'R_EX_btn_e': (-0.001, 0), 'R_EX_pro__L_e': (-0.2, 0), 'R_EX_cys__L_e': (-0.2, 0)}
Oa {'R_EX_thm_e': (-0.001, 0)}


In [ ]:
aux_df.to_csv('auxotrophy_predictions.csv')

# Run memote

============================= test session starts ==============================
platform darwin -- Python 3.10.12, pytest-7.4.3, pluggy-1.3.0
rootdir: /Users/ssulheim
plugins: anyio-4.0.0
collected 146 items / 1 skipped

../../../../anaconda3/envs/memote/lib/python3.10/site-packages/memote/suite/tests/test_annotation.py F [  0%]
FFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFF..         [ 44%]
../../../../anaconda3/envs/memote/lib/python3.10/site-packages/memote/suite/tests/test_basic.py . [ 45%]
....FFF.......F.....FF                                                   [ 60%]
../../../../anaconda3/envs/memote/lib/python3.10/site-packages/memote/suite/tests/test_biomass.py . [ 60%]
.....FF.F                                                                [ 67%]
../../../../anaconda3/envs/memote/lib/python3.10/site-packages/memote/suite/tests/test_consistency.py . [ 67%]
..ssssssssssssssssFFFFFFFF

=================================== FAILURES ================================

KeyError: 'report'

In [ ]:
ml = models['Ml']    

In [ ]:
ml2 = ml.copy()
ml2.remove_reaction('R_PAPSSH')

In [ ]:
reframed.FBA(ml, constraints={'R_EX_cys__L_e': 0, 'R_PAPSSH':0})

Objective: -0.0
Status: Optimal

In [ ]:
# for cs_id in cs_name_to_id.values():
#     ex_id = f'R_EX_{cs_id}_e'
#     if model.reactions[ex_id].lb <= -1:
#         print(f'Change lower bound of {ex_id} from {model.reactions[ex_id].lb} to 0')
#         model.set_flux_bounds(ex_id, lb=0)

In [ ]:
reframed.FBA(model)

Objective: 1.2269581224020234
Status: Optimal

In [ ]:
dfu = test_balance(universe)


In [ ]:
MCC_dict = {}
auxotrophy_dict = {
    'Ml': ['pro__L', 'cys__L']
    }
species_abbr = 'Ct'
print(species_abbr)
model_fn = TMP_FOLDER / f'gapfilled_{species_abbr}.xml'
# model_fn = carveme_draft_folder / f'{species_abbr}.xml'
model = reframed.load_cbmodel(model_fn)
model.reactions['R_EX_cit_e'].lb = 0
# Prep model
print(len(model.reactions), len(model.metabolites))
# curate_model(model)
msg = remove_duplicated_metabolites(model)
print(msg)
print(len(model.reactions), len(model.metabolites))
species_growth_data = binary_growth_data[species_abbr]
if auxotrophy_dict.get(species_abbr):
    auxotrophy_constraints = {}
    for met_id in auxotrophy_dict[species_abbr]:
        auxotrophy_constraints[f'R_EX_{met_id}_e'] =  (-0.2, 0)
else:
    auxotrophy_constraints = {}
test_df, pred_matrix = test_model(model, species_growth_data, cs_name_to_id, min_growth=0.05, additional_constraints=auxotrophy_constraints)
print(test_df)
print(pred_matrix)
print(MCC(pred_matrix))
MCC_dict[species_abbr] = MCC(pred_matrix)

Ct
2548 1720
Deleted these metabolites because they were duplicates: 
2548 1720
Set parameter Username
Academic license - for non-commercial use only - expires 2024-02-26
     Carbon source Carbon source ID  In vitro  In silico Pred outcome
0          Acetate               ac         1   0.242927           TP
1        Adenosine              adn         0   0.000000           TN
2         Cysteine           cys__L         1   0.291688           TP
3      D-arabinose          arab__D         0   0.000000           TN
4        D-Glucose           glc__D         0   0.000000           TN
5         D-xylose           xyl__D         0   0.000000           TN
6          Formate              for         0   0.000000           TN
7         Fumarate              fum         1   0.437636           TP
8        Glutarate           glutar         1   0.292809           TP
9         Glycerol             glyc         0   0.000000           TN
10         Glycine              gly         1   0.285663   

In [ ]:
from operator import attrgetter, itemgetter
from optlang.symbolics import add
from optlang.symbolics import Zero
from optlang.interface import INFEASIBLE, OPTIMAL


def check_stoichiometric_consistency(model):
    """
    Verify the consistency of the model's stoichiometry.

    Parameters
    ----------
    model : cobra.Model
        The metabolic model under investigation.

    Notes
    -----
    See [1]_ section 3.1 for a complete description of the algorithm.

    .. [1] Gevorgyan, A., M. G Poolman, and D. A Fell.
           "Detection of Stoichiometric Inconsistencies in Biomolecular
           Models."
           Bioinformatics 24, no. 19 (2008): 2245.

    """
    problem = model.problem
    # The transpose of the stoichiometric matrix N.T in the paper.
    stoich_trans = problem.Model()
    # We clone the existing configuration in order to apply non-default
    # settings, for example, for solver verbosity or timeout.
    stoich_trans.configuration = problem.Configuration.clone(
        config=model.solver.configuration, problem=stoich_trans
    )
    internal_rxns = get_internals(model)
    metabolites = set(met for rxn in internal_rxns for met in rxn.metabolites)
    print("model '%s' has %d internal reactions", model.id, len(internal_rxns))
    print("model '%s' has %d internal metabolites", model.id, len(metabolites))

    stoich_trans.add([problem.Variable(m.id, lb=1) for m in metabolites])
    stoich_trans.update()
    add_reaction_constraints(
        stoich_trans, internal_rxns, problem.Constraint
    )
    # The objective is to minimize the metabolite mass vector.
    stoich_trans.objective = problem.Objective(Zero, direction="min", sloppy=True)
    stoich_trans.objective.set_linear_coefficients(
        {var: 1.0 for var in stoich_trans.variables}
    )
    status = stoich_trans.optimize()
    if status == OPTIMAL:
        return True
    elif status == INFEASIBLE:
        return False
    else:
        raise RuntimeError(
            "Could not determine stoichiometric consistencty."
            " Solver status is '{}'"
            " (only optimal or infeasible expected).".format(status)
        )
def get_internals(model):
    """
    Return non-boundary reactions and their metabolites.

    Boundary reactions are unbalanced by their nature. They are excluded here
    and only the metabolites of the others are considered.

    Parameters
    ----------
    model : cobra.Model
        The metabolic model under investigation.

    """
    biomass = set(find_biomass_reaction(model))
    if len(biomass) == 0:
        print(
            "No biomass reaction detected. Consistency test results "
            "are unreliable if one exists."
        )
    return set(model.reactions) - (set(model.boundary) | biomass)

def find_biomass_reaction(model):
    """
    Return a list of the biomass reaction(s) of the model.

    Identifiy possible biomass reactions using multiple steps:
    1. Look for candidate reactions that include the SBO term ``SBO:0000629``
    for biomass production,
    2. the 'buzzwords' biomass, growth, and bof in reaction names
    and identifiers,
    3. reactions that involve a metabolite with the SBO term ``SBO:0000649``
    for biomass,
    4. or reactions that involve a metabolite whose name or identifier
    contains the 'buzzword' biomass.
    Return identified reactions excluding any boundary reactions.

    Parameters
    ----------
    model : cobra.Model
        The metabolic model under investigation.

    Returns
    -------
    list
        Identified biomass reactions (if any).

    """
    boundary = frozenset(model.boundary)

    # 1.
    candidates = {r for r in model.reactions if filter_sbo_term(r, "SBO:0000629")}
    candidates.difference_update(boundary)
    if candidates:
        return sorted(candidates, key=attrgetter("id"))

    # 2.
    name_buzzwords = (
        re.compile(r"\bbiomass"),
        re.compile(r"\bgrowth"),
        re.compile(r"bof"),
    )
    id_buzzwords = ("biomass",)
    candidates = {
        r
        for r in model.reactions
        if filter_match_name(r, name_buzzwords) or filter_identifier(r, id_buzzwords)
    }
    candidates.difference_update(boundary)
    if candidates:
        return sorted(candidates, key=attrgetter("id"))

    # 3.
    name_buzzwords = (re.compile(r"\bbiomass"),)
    id_buzzwords = ("biomass",)
    sbo_metabolites = {
        m for m in model.metabolites if filter_sbo_term(m, "SBO:0000649")
    }
    metabolites = {
        m
        for m in model.metabolites
        if filter_match_name(m, name_buzzwords) or filter_identifier(m, id_buzzwords)
    }
    # Many metabolites may match 'SBO:0000649', we filter those further by name
    # and ID.
    sbo_metabolites.intersection_update(metabolites)
    if sbo_metabolites:
        candidates = {r for m in sbo_metabolites for r in m.reactions}
        candidates.difference_update(boundary)
        if candidates:
            return sorted(candidates, key=attrgetter("id"))

    # 4.
    candidates = {r for m in metabolites for r in m.reactions}
    candidates.difference_update(boundary)
    if candidates:
        return sorted(candidates, key=attrgetter("id"))

    return []

def filter_sbo_term(component, sbo_term):
    """
    Return true if the component is annotated with the given SBO term.

    Parameters
    ----------
    component : cobra.Reaction or cobra.Metabolite
        Either a reaction or a metabolite instance.
    sbo_term : str
        The term for either biomass production or biomass.

    """
    return component.annotation.get("sbo", "") == sbo_term
def filter_match_name(component, buzzwords):
    """
    Return whether the component's name matches a biomass description.

    Notes
    -----
    Regex patterns are necessary here to prevent, for example, 'non-growth' from
    matching.

    Parameters
    ----------
    component : cobra.Reaction or cobra.Metabolite
        Either a reaction or a metabolite instance.
    buzzwords : collection of regex patterns
        One or more regular expression patterns to match against the name of the
        component.

    Returns
    -------
    bool
        True if there was any match at all.

    """
    if component.name is None:
        return False
    name = component.name.lower()
    return any(b.match(name) for b in buzzwords)


def filter_identifier(component, buzzwords):
    """
    Return whether the component's identifier contains a biomass description.

    Notes
    -----
    We check substring presence here because identifiers are often prefixed with
    ``M_`` or ``R_``.

    Parameters
    ----------
    component : cobra.Reaction or cobra.Metabolite
        Either a reaction or a metabolite instance.
    buzzwords : iterable of str
        One or more buzzwords that the identifier should contain.

    Returns
    -------
    bool
        True if there was any match at all.

    """
    identifier = component.id.lower()
    return any(b in identifier for b in buzzwords)

def add_reaction_constraints(model, reactions, Constraint):
    """
    Add the stoichiometric coefficients as constraints.

    Parameters
    ----------
    model : optlang.Model
        The transposed stoichiometric matrix representation.
    reactions : iterable
        Container of `cobra.Reaction` instances.
    Constraint : optlang.Constraint
        The constraint class for the specific interface.

    """
    constraints = []
    for rxn in reactions:
        expression = add(
            [c * model.variables[m.id] for m, c in rxn.metabolites.items()]
        )
        constraints.append(Constraint(expression, lb=0, ub=0, name=rxn.id))
    model.add(constraints)

In [ ]:
r = modelr.reactions.R_ACALD

In [ ]:
r.gpr

(G_WP_302089245_1 or G_WP_302089757_1 or G_WP_302091681_1)

In [ ]:
j = 0
for r in modelr.reactions.values():
    if r.id.startswith('R_EX_'):
        continue
    if not r.gpr:
        j+= 1


In [ ]:
modelr.add_metabolite?

Signature: modelr.add_metabolite(metabolite, replace=True)
Docstring:
Add a metabolite to the model.

Arguments:
    metabolite (Metabolite): metabolite to add
    replace (bool): replace previous metabolite with same id (default: True)
File:      ~/anaconda3/lib/python3.11/site-packages/reframed/core/model.py
Type:      method

In [ ]:
modelr.metabolites.M_glc__D_e.metadata

OrderedDict([('FORMULA', 'C6H12O6'),
             ('CHARGE', '0'),
             ('SBOTerm', 'SBO:0000247'),
             ('kegg.compound', 'C00031'),
             ('chebi', 'CHEBI:4167'),
             ('kegg.drug', 'D00009'),
             ('hmdb', 'HMDB06564'),
             ('biocyc', 'META:Glucopyranose'),
             ('metanetx.chemical', 'MNXM41'),
             ('inchikey', 'WQZGKKKJIJFFOK-GASJEMHNSA-N'),
             ('seed.compound', 'cpd26821')])

In [ ]:
import cobra
model = cobra.io.read_sbml_model(model_fn)


In [ ]:
check_stoichiometric_consistency(model)

model '%s' has %d internal reactions Ct 2266
model '%s' has %d internal metabolites Ct 1711


False

In [ ]:
modelr = reframed.load_cbmodel(model_fn)

In [ ]:
df = test_balance(modelr)

In [ ]:
df.loc[df['mass_balance']!=True, :]

,r_id,mass_balance,mass,element_balance,element_dict,charge_balance,charge
572,R_CMCBTFL,False,-0.001008,False,"{'C': 0.0, 'H': 1.0, 'N': 0.0, 'O': 0.0, 'Fe':...",False,NaN
574,R_CMCBTFabcpp,False,-0.001008,False,"{'C': 0.0, 'H': 1.0, 'N': 0.0, 'O': 0.0, 'P': ...",False,NaN
708,R_DHBSZ3FEabcpp,False,-0.001008,False,"{'C': 0.0, 'H': 1.0, 'N': 0.0, 'O': 0.0, 'P': ...",False,NaN
1997,R_SALCHS4FEabcpp,False,-0.001008,False,"{'C': 0.0, 'H': 1.0, 'N': 0.0, 'O': 0.0, 'P': ...",False,NaN
2080,R_TAGabc,False,-0.001008,False,"{'C': 0.0, 'H': 1.0, 'N': 0.0, 'O': 0.0, 'P': ...",False,1.0


In [ ]:
problem = model.problem
# The transpose of the stoichiometric matrix N.T in the paper.
stoich_trans = problem.Model()
# We clone the existing configuration in order to apply non-default
# settings, for example, for solver verbosity or timeout.
stoich_trans.configuration = problem.Configuration.clone(
    config=model.solver.configuration, problem=stoich_trans
)
internal_rxns = get_internals(model)
metabolites = set(met for rxn in internal_rxns for met in rxn.metabolites)
print("model '%s' has %d internal reactions", model.id, len(internal_rxns))
print("model '%s' has %d internal metabolites", model.id, len(metabolites))

stoich_trans.add([problem.Variable(m.id, lb=1) for m in metabolites])
stoich_trans.update()
add_reaction_constraints(
    stoich_trans, internal_rxns, problem.Constraint
)
# The objective is to minimize the metabolite mass vector.
stoich_trans.objective = problem.Objective(Zero, direction="min", sloppy=True)
stoich_trans.objective.set_linear_coefficients(
    {var: 1.0 for var in stoich_trans.variables}
)
status = stoich_trans.optimize()

model '%s' has %d internal reactions Ct 2266
model '%s' has %d internal metabolites Ct 1711


In [ ]:
status

'infeasible'

In [ ]:
exchanges = model.exchanges

In [ ]:
noE = [r for r in model.reactions if not r in exchanges]

In [ ]:
[x for x in noE if not x in internals]

[<Reaction sink_2ohph_c at 0x2a2b83a50>,
 <Reaction sink_4crsol_c at 0x2a2b838d0>,
 <Reaction sink_amob_c at 0x2a2b90bd0>,
 <Reaction sink_hemeO_c at 0x2a2b90d50>,
 <Reaction sink_mobd_c at 0x2a2b79590>,
 <Reaction sink_sheme_c at 0x2a2b90fd0>,
 <Reaction Growth at 0x2a2c513d0>,
 <Reaction EX_pectin_e at 0x2a2c66510>]

In [ ]:
k = 0
for m in model.metabolites:
    if m.id[-2:] == '_c':
        k+=1
print(k)

1118


In [ ]:
def test_leaky(model):
    model = model.copy()
    exchanges = model.get_exchange_reactions()
    for r_id in exchanges:
        model.reactions[r_id].lb = 0

    for m in model.metabolites.values():
        if m.id[-2:] == '_c':
            r_id = f"R_DM_{m.id.lstrip('M_')}"
            reaction_string = f"{r_id}: {m.id} <->  [0, 1000]"
            model.add_reaction_from_str(reaction_string)
            sol = reframed.FBA(model, objective=r_id)
            if sol.status == Status.OPTIMAL:
                print(sol)
                if sol.fobj > 1e-6:
                    print(m.id, sol)
            

In [ ]:
solution.show_values('R_EX_')

R_EX_h2o_e    1.8
R_EX_o2_e    -0.9


In [ ]:
MCC_dict = {}
auxotrophy_dict = {
    'Ml': ['pro__L', 'cys__L']
    }
for species_abbr in ['At',  'Ct', 'Oa', 'Ml']:#,'At',  'Ct', 'Oa', 
    print(species_abbr)
    model_fn = TMP_FOLDER / f'gapfilled_{species_abbr}.xml'
    # model_fn = carveme_draft_folder / f'{species_abbr}.xml'
    model = reframed.load_cbmodel(model_fn)
    # Prep model
    curate_model(model)
    species_growth_data = binary_growth_data[species_abbr]
    if auxotrophy_dict.get(species_abbr):
        auxotrophy_constraints = {}
        for met_id in auxotrophy_dict[species_abbr]:
            auxotrophy_constraints[f'R_EX_{met_id}_e'] =  (-0.2, 0)
    else:
        auxotrophy_constraints = {}

    test_df, pred_matrix = test_model(model, species_growth_data, cs_name_to_id, min_growth=0.05, additional_constraints=auxotrophy_constraints)
    print(test_df)
    print(pred_matrix)
    print(MCC(pred_matrix))
    MCC_dict[species_abbr] = MCC(pred_matrix)

In [ ]:
reframed.FBA(model, constraints={'R_EX_glc__D_e':0, 'R_EX_pro__L_e':-0.1,  'R_EX_cys__L_e':-0.1})


# Verify auxotrophies / fix
Procedure to fix auxotrophy:
1. See if there are any reactions that becomes essentials when the metabolite the strain is auxotrophic for is not present and compare it with the list of all essential reactions. If this works choose one (not transport or exchange) of these reactions
2. If not, find all reactions that produces the compound. Get all the non-transport reactions and see if they fix the auxotrophy if they are all removed. Then remove one by one until no more reactions can be removed.
3. Verify that this hasn't affected the model accuracy




In [ ]:
sorted_df = test_df.loc[test_df['Pred outcome'] == 'TP', :].sort_values(by='In silico', ascending = False)

In [ ]:
test_met = sorted_df.iloc[5]['Carbon source ID']
# while test_met is not one of the auxotrophie mets

In [ ]:
auxotrophy_constraints

{'R_EX_pro__L_e': -0.2, 'R_EX_cys__L_e': -0.2}

In [ ]:
zero_constraints = {f'R_EX_{test_met}_e': -10}
# zero_constraints['R_PAPSR'] = 0

In [ ]:
zero_solution = reframed.FBA(model, constraints = zero_constraints)

In [ ]:
zero_solution

Objective: -0.0
Status: Optimal

In [ ]:
except_one_met_results = {}
for r_id, uptake in auxotrophy_constraints.items():
    except_one_met_constraints = zero_constraints.copy()
    except_one_met_constraints.update({key: value for key, value in auxotrophy_constraints.items() if key != r_id})
    # for key in ['R_CYSS', 'R_CYSTGL']:
    #     except_one_met_constraints[key]=0
    
    print(except_one_met_constraints)
    except_one_met_results[r_id] = reframed.FBA(model, constraints = except_one_met_constraints).fobj

except_one_met_results

{'R_EX_sucr_e': -10, 'R_EX_cys__L_e': (-0.2, 0)}
{'R_EX_sucr_e': -10, 'R_EX_pro__L_e': (-0.2, 0)}


{'R_EX_pro__L_e': -0.0, 'R_EX_cys__L_e': -0.0}

{'R_EX_pro__L_e': -0.0, 'R_EX_cys__L_e': -0.0}

In [ ]:
sol = reframed.FBA(model, constraints = except_one_met_constraints)
c = all_mets_constraints.copy()
c['R_CYSabc'] = 0
# reframed.FBA(model, constraints = c)

In [ ]:
reframed.FBA(model, constraints = {'R_EX_glc__D_e': (-10, 0), 'R_EX_pro__L_e': (-0.1, 0), 'R_EX_cys__L_e': (-0.1, 0)})

Objective: 0.45237610549410784
Status: Optimal

In [ ]:
sol.show_values('R_CYS*')

R_CYSTGL      0.0724495
R_CYSTS       0.0724495
R_CYTBD2      2.28493
R_CYTK1       0.864717
R_CYTBO3_4pp  17.7151


In [ ]:
for r_id, value in except_one_met_results.items():
    if value > 0.1:
        print(r_id)
        # Not auxotrophic for this met - Fix!
        except_one_met_constraints = zero_constraints.copy()
        except_one_met_constraints.update({key: value for key, value in auxotrophy_constraints.items() if key != r_id})
        except_one_essentials = reframed.essential_reactions(model, constraints=except_one_met_constraints)

        # all_mets_constraints = zero_constraints.copy()
        # all_mets_constraints.update(auxotrophy_constraints)
        # all_mets_essentials = reframed.essential_reactions(model, constraints = all_mets_constraints)

R_EX_cys__L_e


In [ ]:
targets = [r for r in except_one_essentials if not r in essential_reactions]

In [ ]:
targets2 = [r for r in producers if not r in all_mets_essentials]

In [ ]:
r = model.reactions['R_AHSERL4']

In [ ]:
r.metadata

OrderedDict([('SBOTerm', 'SBO:0000176'),
             ('XMLAnnotation',
              '<annotation xmlns:sbml="http://www.sbml.org/sbml/level3/version1/core">\n  <rdf:RDF xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#" xmlns:dcterms="http://purl.org/dc/terms/" xmlns:vCard="http://www.w3.org/2001/vcard-rdf/3.0#" xmlns:vCard4="http://www.w3.org/2006/vcard/ns#" xmlns:bqbiol="http://biomodels.net/biology-qualifiers/" xmlns:bqmodel="http://biomodels.net/model-qualifiers/">\n    <rdf:Description rdf:about="#R_AHSERL4">\n      <bqbiol:is>\n        <rdf:Bag>\n          <rdf:li rdf:resource="http://identifiers.org/ec-code/2.5.1.47"/>\n          <rdf:li rdf:resource="http://identifiers.org/ec-code/2.5.1.49"/>\n          <rdf:li rdf:resource="http://identifiers.org/ec-code/4.2.99.8"/>\n          <rdf:li rdf:resource="http://identifiers.org/metanetx.reaction/MNXR95636"/>\n          <rdf:li rdf:resource="http://identifiers.org/kegg.reaction/R04859"/>\n          <rdf:li rdf:resource="http://

In [ ]:
fne = TMP_FOLDER / 'essential_reactions_Ml.json'
with open(fne, 'r') as f:
    essential_reactions = json.load(f)
essential_reactions = essential_reactions['essential_reactions']

In [ ]:
reframed.ReactionType.OTHER

<ReactionType.OTHER: 'other'>

In [ ]:
except_one_met_constraints

{'R_EX_adn_e': -10, 'R_EX_pro__L_e': -0.2}

In [ ]:
producers = model.get_metabolite_producers('M_cys__L_c')

In [ ]:
producers

['R_AHSERL4',
 'R_AMPTASECG',
 'R_CYSS',
 'R_CYSTGL',
 'R_CYSabc',
 'R_CYSabcpp',
 'R_GLYCYSAP']

In [ ]:
model.reactions['R_CYSS'].reaction_type

<ReactionType.ENZYMATIC: 'enzymatic'>

In [ ]:
model.get_exchange_reactions().pop(0)

'R_sink_2ohph_c'

In [ ]:

print("###")
for r_id in targets:
    print(model.reactions[r_id])

###


In [ ]:
all_mets_constraints = zero_constraints.copy()
all_mets_constraints.update(auxotrophy_constraints)
reframed.FBA(model, constraints = all_mets_constraints)

Objective: 0.7838349962837273
Status: Optimal

In [ ]:
a = {'1':1}
b = a.update({'2':2})

In [ ]:
auxotrophy_constraints

{'R_EX_pro__L_e': -0.2, 'R_EX_cys__L_e': -0.2}

In [ ]:
def verify_auxotrophies(species_abbr, model, auxotrophy_constraints, growth_constraints, method = 'FBA', min_growth = 0.1):
    solution = reframed.FBA(model, constraints = growth_constraints)
    logger = logging.getLogger(f'auxotrophy.{species_abbr}')
    complete_constraints = growth_constraints.copy()
    complete_constraints.update(auxotrophy_constraints)
    if solution.fobj > min_growth:
        logger.info(f'{species_abbr} grows on {growth_constraints.keys()[0]} without {auxotrophy_constraints.keys()[0]}')
        essential_rxns = reframed.essential_reactions(model, min_growth=min_growth, constraints=growth_constraints)
        essential_rxns_wa = reframed.essential_reactions(model, min_growth=min_growth, constraints=complete_constraints)
        potential_reactions_to_remove = [r_id for r_id in essential_reactions if r_id not in essential_rxns_wa]
        logger.info('Possible reactions to remove to fix auxotrophy: %s', potential_reactions_to_remove)
        

Set parameter Username
Academic license - for non-commercial use only - expires 2024-02-26


In [ ]:
test_results

(     Carbon source Carbon source ID  In vitro  In silico Pred outcome
 0          Acetate               ac         1   0.248249           TP
 1        Adenosine              adn         1   0.527347           TP
 2         Cysteine           cys__L         1   0.376903           TP
 3      D-arabinose          arab__D         1   0.698863           TP
 4        D-Glucose           glc__D         1   0.782571           TP
 5         D-xylose           xyl__D         1   0.682878           TP
 6          Formate              for         1   0.220570           TP
 7         Fumarate              fum         1   0.420471           TP
 8        Glutarate           glutar         1   0.364161           TP
 9         Glycerol             glyc         1   0.589540           TP
 10         Glycine              gly         0   0.000000           TN
 11         Inosine              ins         1   0.501973           TP
 12       L-Alanine           ala__L         1   0.375162           TP
 13   

In [ ]:
model.get_metabolite_reactions('M_fe3dcit_e')

['R_FE3DCITexs', 'R_FE3DCITtonex', 'R_EX_fe3dcit_e']

In [ ]:
model.reactions['R_FE3DCITtonex']

R_FE3DCITtonex: M_fe3dcit_e + M_h_p --> M_fe3dcit_p + M_h_c

In [ ]:
for comp in model.compartments.values():
    print()

c cytosol
p periplasm
e extracellular space


In [ ]:
for key, name in model.compartments.items():
    print(key, name)

c cytosol
p periplasm
e extracellular space


In [ ]:
reframed.save_cbmodel(model, 'test.xml')

In [ ]:
comp.__dict__

{'id': 'C_e',
 'name': 'extracellular space',
 'size': 1.0,
 'external': True,
 'metadata': OrderedDict([('SBOTerm', 'SBO:0000290')])}